# 01 · Ingesta y base de datos en la nube

Este notebook cubre dos cosas que conviene no confundir.

| | Qué es | ¿Es parte de la medallion? |
|---|---|---|
| **Parte A · Seeding** | Poblar el sistema operacional del banco desde Kaggle | **No.** Es aprovisionamiento, ocurre una sola vez |
| **Parte B · Bronze** | Primera extracción JDBC hacia el lakehouse | **Sí.** Aquí empieza el pipeline |

El enunciado define Bronze como *«copia inmutable de la **extracción JDBC**»* — no como copia
del CSV. Por eso el archivo de Kaggle nunca entra al lakehouse: entra a Postgres, y el
lakehouse lee de Postgres.

En un banco real la Parte A no existiría: `customers_raw` ya estaría poblada por los sistemas
del banco. Kaggle está haciendo de sustituto de esa realidad, y se documenta como tal.

## Entorno

In [0]:
# pg8000: driver Postgres en Python puro (psycopg[binary] tumba el kernel serverless).
# protobuf < 6: el upgrade del SDK arrastra protobuf 6 y rompe Spark.
%pip install --quiet --upgrade "databricks-sdk>=0.81.0" "pg8000" "kagglehub" "protobuf>=5.26.1,<6"
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import io
import os
import ssl
import hashlib
import datetime as dt

import pandas as pd
import pg8000.dbapi
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# --- Lakebase resources -------------------------------------------------------
PROJECT_ID = "bank-churn-prediction"
BRANCH     = next(iter(w.postgres.list_branches(f"projects/{PROJECT_ID}"))).name
ENDPOINT   = next(iter(w.postgres.list_endpoints(BRANCH))).name

ep         = w.postgres.get_endpoint(ENDPOINT)
PGHOST     = ep.status.hosts.host
PGDATABASE = "databricks_postgres"
PGPORT     = 5432
PGUSER     = w.current_user.me().user_name
SCHEMA     = "bank_churn"

# --- Kaggle source ------------------------------------------------------------
KAGGLE_DATASET = "mathchi/churn-for-bank-customers"
KAGGLE_URL     = f"https://www.kaggle.com/datasets/{KAGGLE_DATASET}"

print("endpoint:", ENDPOINT)
print("host    :", PGHOST)
print("user    :", PGUSER)

endpoint: projects/bank-churn-prediction/branches/production/endpoints/primary
host    : ep-calm-art-d80j4anc.database.us-east-2.cloud.databricks.com
user    : <usuario>


In [0]:
def connect():
    """Nueva conexión con credencial OAuth fresca (60 min de vida)."""
    token = w.postgres.generate_database_credential(ENDPOINT).token
    return pg8000.dbapi.connect(
        host=PGHOST, port=PGPORT, database=PGDATABASE,
        user=PGUSER, password=token,
        ssl_context=ssl.create_default_context(),
    )


def query(sql, params=None):
    """Ejecuta SELECT y devuelve DataFrame."""
    with connect() as conn:
        cur = conn.cursor()
        cur.execute(sql, params or ())
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


# Smoke test
print(query("SELECT current_database() AS db, current_user AS usr").to_string(index=False))

                 db                 usr
databricks_postgres <usuario>


---

# Parte A · Seeding del sistema operacional

*Fuera de la arquitectura medallion. Se ejecuta una sola vez.*

## A1 · Descarga y trazabilidad

El enunciado exige registrar **por código** el enlace, la fecha de descarga, el nombre del
archivo, su tamaño y el número de filas y columnas. Nada de esto se transcribe a mano.

Se añade también un hash SHA-256: si alguien reejecuta el notebook meses después y el hash
cambia, el dataset de origen se modificó. Es la forma barata de detectar que la fuente ya no
es la misma.

In [0]:
import kagglehub

# Requiere credenciales de Kaggle. En Databricks, la via mas simple es definirlas
# como variables de entorno antes de descargar (Account -> Settings -> Create New Token).
# Si ya hay un token en ~/.kaggle/kaggle.json, esta celda no pide nada.
#   os.environ["KAGGLE_USERNAME"] = dbutils.widgets.get("kaggle_user")
#   os.environ["KAGGLE_KEY"]      = dbutils.widgets.get("kaggle_key")

path = kagglehub.dataset_download(KAGGLE_DATASET)
print("descargado en:", path)
print("contenido    :", os.listdir(path))

descargado en: /home/spark-3811c5a0-7cc1-4e6d-a391-c6/.cache/kagglehub/datasets/mathchi/churn-for-bank-customers/versions/1
contenido    : ['churn.csv']


In [0]:
csv_path = os.path.join(path, "churn.csv")

raw = pd.read_csv(csv_path)

with open(csv_path, "rb") as fh:
    sha256 = hashlib.sha256(fh.read()).hexdigest()

TRACE = {
    "source_url":     KAGGLE_URL,
    "source_file":    os.path.basename(csv_path),
    "downloaded_at":  dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "size_bytes":     os.path.getsize(csv_path),
    "rows":           int(raw.shape[0]),
    "columns":        int(raw.shape[1]),
    "sha256":         sha256,
}

for k, v in TRACE.items():
    print(f"{k:>14}: {v}")

print("\ncolumnas del origen:")
print(list(raw.columns))

    source_url: https://www.kaggle.com/datasets/mathchi/churn-for-bank-customers
   source_file: churn.csv
 downloaded_at: 2026-08-01T22:59:35+00:00
    size_bytes: 684858
          rows: 10000
       columns: 14
        sha256: 3996cd1fa372e0db0cd9c0ebac35bbd4e8e3c65fb942bb010c826e7b1eeef0a0

columnas del origen:
['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


## A2 · Verificaciones de calidad (V1–V8)

Ocho comprobaciones acordadas antes de crear las tablas con restricciones. Las cotas de los
`CHECK` en `02_modeled.sql` son **expectativas**, no hechos: esta celda las convierte en
hechos o las corrige.

Que una verificación falle no es un problema — es un hallazgo documentable para el informe.

In [0]:
checks = {}

# V1 · customer_id único
checks["V1 customer_id único"] = (
    raw["CustomerId"].is_unique,
    f"{raw['CustomerId'].nunique()} distintos de {len(raw)} filas",
)

# V2 · filas y columnas (ya en TRACE)
checks["V2 forma del dataset"] = (True, f"{raw.shape[0]} filas x {raw.shape[1]} columnas")

# V3 · dominios categóricos
geo = sorted(raw["Geography"].unique())
gen = sorted(raw["Gender"].unique())
checks["V3 dominios categóricos"] = (
    set(geo) == {"France", "Spain", "Germany"} and set(gen) == {"Male", "Female"},
    f"Geography={geo} | Gender={gen}",
)

# V4 · rangos numéricos reales
ranges = {c: (raw[c].min(), raw[c].max())
          for c in ["CreditScore", "Age", "Tenure", "Balance",
                    "NumOfProducts", "EstimatedSalary"]}
checks["V4 rangos numéricos"] = (True, " | ".join(f"{c}[{lo}, {hi}]" for c, (lo, hi) in ranges.items()))

# V5 · nulos
nulls = raw.isna().sum()
checks["V5 sin nulos"] = (nulls.sum() == 0, f"total nulos = {int(nulls.sum())}")

# V6 · divisiones por cero en las features de ratio
zero_salary = int((raw["EstimatedSalary"] == 0).sum())
age_18      = int((raw["Age"] == 18).sum())
checks["V6 riesgo división por cero"] = (
    zero_salary == 0 and age_18 == 0,
    f"EstimatedSalary=0 -> {zero_salary} | Age=18 -> {age_18}",
)

# V7 · duplicados de fila completa (ignorando RowNumber)
dups = int(raw.drop(columns=["RowNumber"]).duplicated().sum())
checks["V7 sin duplicados"] = (dups == 0, f"{dups} filas duplicadas")

# V8 · coherencia tenure <= age - 18
incoherent = int((raw["Tenure"] > raw["Age"] - 18).sum())
checks["V8 tenure coherente"] = (incoherent == 0, f"{incoherent} filas incoherentes")

print(f"{'':<28} {'':<5} detalle")
print("-" * 100)
for name, (ok, detail) in checks.items():
    print(f"{name:<28} {'OK ' if ok else 'REV'}  {detail}")

                                   detalle
----------------------------------------------------------------------------------------------------
V1 customer_id unico         OK   10000 distintos de 10000 filas
V2 forma del dataset         OK   10000 filas x 14 columnas
V3 dominios categoricos      OK   Geography=['France', 'Germany', 'Spain'] | Gender=['Female', 'Male']
V4 rangos numericos          OK   CreditScore[350, 850] | Age[18, 92] | Tenure[0, 10] | Balance[0.0, 250898.09] | NumOfProducts[1, 4] | EstimatedSalary[11.58, 199992.48]
V5 sin nulos                 OK   total nulos = 0
V6 riesgo division por cero  REV  EstimatedSalary=0 -> 0 | Age=18 -> 22
V7 sin duplicados            OK   0 filas duplicadas
V8 tenure coherente          REV  322 filas incoherentes


### A2.1 · Diagnóstico de V6 y V8

Dos comprobaciones salieron marcadas para revisión. Ninguna se resuelve borrando filas: primero
hay que entender qué significan.

**V6 — 22 clientes con `Age = 18`.** No es un error: son clientes legítimos. El problema es
aritmético — la feature prevista `tenure_age_ratio = tenure / (age - 18)` divide entre cero
para ellos.

**V8 — 322 filas con `tenure > age - 18`.** Aquí conviene ser preciso: esa comprobación codifica
un supuesto *propio* — que una relación bancaria empieza a los 18 años — y ese supuesto es
discutible, porque existen cuentas junior y cuentas abiertas por los padres. El hallazgo no es
«hay 322 registros corruptos», sino «la regla escrita no describe estos datos».

La celda siguiente responde tres preguntas antes de decidir nada: cuán extremas son esas filas,
si el grupo se comporta distinto, y si existe alguna relación real entre edad y antigüedad.

In [0]:
inc = raw[raw["Tenure"] > raw["Age"] - 18]

print(f"filas incoherentes: {len(inc)}  ({len(inc)/len(raw):.1%})")
print("\nedad de esos clientes:")
print(inc["Age"].describe()[["min", "25%", "50%", "max"]])

# edad implícita al abrir la cuenta
implied = inc["Age"] - inc["Tenure"]
print(f"\nedad implícita de apertura — min: {implied.min()}, mediana: {implied.median()}")
print(f"aperturas antes de nacer: {(implied < 0).sum()}")

# tasa de abandono: ¿este grupo se comporta distinto?
print(f"\nabandono global      : {raw['Exited'].mean():.1%}")
print(f"abandono incoherentes: {inc['Exited'].mean():.1%}")

# la prueba decisiva
print(f"\ncorrelación Age–Tenure (Spearman): {raw['Age'].corr(raw['Tenure'], method='spearman'):.4f}")

filas incoherentes: 322  (3.2%)

edad de esos clientes:
min    18.0
25%    21.0
50%    22.0
max    27.0
Name: Age, dtype: float64

edad implícita de apertura — min: 8, mediana: 15.0
aperturas antes de nacer: 0

abandono global      : 20.4%
abandono incoherentes: 7.5%

correlación Age–Tenure (Spearman): -0.0104


Dos cosas destacan de la salida anterior.

**Ninguna apertura ocurre antes del nacimiento**: la edad implícita de apertura va de 8 a 17
años. Son valores implausibles bajo el supuesto original, pero no imposibles.

Y la **correlación de Spearman entre `Age` y `Tenure` es −0,0104**, prácticamente cero. En datos
bancarios reales esa correlación es positiva: a más edad, más tiempo disponible para acumular
antigüedad. Aquí no existe relación alguna, lo que indica que ambas variables se generaron de
forma independiente. Las 322 filas son la consecuencia aritmética de sortear `Tenure` sin mirar
`Age`.

Queda un dato que parece un hallazgo y muy probablemente sea una **trampa**: el grupo incoherente
abandona un 7,5% frente al 20,4% global. Pero esas 322 filas tienen todas entre 18 y 27 años, y
el abandono crece con la edad. La incoherencia y el bajo abandono comparten una causa común —la
edad— sin que una explique la otra.

La celda siguiente aísla el confusor: compara coherentes contra incoherentes **dentro del mismo
tramo de edad**.

In [0]:
jovenes = raw[raw["Age"].between(18, 27)]
coh = jovenes[jovenes["Tenure"] <= jovenes["Age"] - 18]
inc = jovenes[jovenes["Tenure"] >  jovenes["Age"] - 18]

print(f"jóvenes coherentes   : {coh['Exited'].mean():.1%}  (n={len(coh)})")
print(f"jóvenes incoherentes : {inc['Exited'].mean():.1%}  (n={len(inc)})")

jóvenes coherentes   : 7.0%  (n=698)
jóvenes incoherentes : 7.5%  (n=322)


7,0% frente a 7,5%, con n=698 y n=322. Medio punto de diferencia es ruido.

**Confirmado: la incoherencia no aporta información.** Todo era edad.

Esto descarta con evidencia la tentación de crear una variable `es_incoherente`. En validación
parecería predictiva, pero solo sería un proxy ruidoso de «cliente joven» — el tipo de feature
espuria que funciona en el notebook y falla en producción.

Queda entonces la pregunta de fondo sobre `Tenure`: si no se relaciona con la edad, ¿se relaciona
al menos con el abandono?

In [0]:
print("Tenure vs Exited :", raw["Tenure"].corr(raw["Exited"], method="spearman").round(4))
print("Age    vs Exited :", raw["Age"].corr(raw["Exited"], method="spearman").round(4))

Tenure vs Exited : -0.014
Age    vs Exited : 0.324


`Tenure` da −0,014 frente al 0,324 de `Age`. Pero conviene no leerlo mal: **una correlación
cercana a cero no significa «no hay relación», sino «no hay relación monótona»**.

Si el abandono fuese alto con antigüedad 0 (cliente recién llegado que se arrepiente), bajo en el
medio y alto de nuevo en 10, Spearman daría ~0 y aun así existiría una señal en forma de U
perfectamente aprovechable por un árbol.

`Tenure` es discreta con once valores, así que la comprobación por grupos es **definitiva**, no
aproximada. Y conviene hacer la simétrica con la edad: un coeficiente de 0,324 podría estar
ocultando una relación bastante más fuerte si no es monótona.

In [0]:
print(raw.groupby("Tenure")["Exited"].agg(tasa="mean", n="count").assign(
      tasa=lambda d: (d["tasa"]*100).round(1)))

        tasa     n
Tenure            
0       23.0   413
1       22.4  1035
2       19.2  1048
3       21.1  1009
4       20.5   989
5       20.7  1012
6       20.3   967
7       17.2  1028
8       19.2  1025
9       21.6   984
10      20.6   490


In [0]:
raw["_age_band"] = pd.cut(raw["Age"], [17,30,40,50,60,100],
                          labels=["18-30","31-40","41-50","51-60","60+"])
print(raw.groupby("_age_band", observed=True)["Exited"].agg(tasa="mean", n="count").assign(
      tasa=lambda d: (d["tasa"]*100).round(1)))
raw.drop(columns="_age_band", inplace=True)

           tasa     n
_age_band            
18-30       7.5  1968
31-40      12.1  4451
41-50      34.0  2320
51-60      56.2   797
60+        24.8   464


### Conclusiones del diagnóstico

**`Tenure` es ruido.** Las tasas van de 17,2% a 23,0% sin forma ni tendencia. Con ~1.000 casos
por grupo y una tasa base de 20,4%, el error estándar ronda 1,3 puntos: toda la variación
observada cabe dentro del ruido muestral. Se formalizará con una prueba chi-cuadrado de
independencia en el notebook 03 — un test que *no* rechaza la hipótesis nula es un análisis
inferencial válido, y más honesto que reportar solo los que confirman lo que uno espera.

Detalle adicional: `tenure = 0` y `tenure = 10` tienen la mitad de casos que el resto (413 y 490
frente a ~1.000). Es la huella de un sorteo uniforme redondeado — otra señal de generación
sintética.

**`Age` describe una U invertida y es el predictor dominante.** El abandono sube hasta el 56,2%
en el tramo 51-60 y **cae al 24,8%** por encima de los 60. La caída es real: con n=464 el error
estándar es de 2 puntos.

Tres consecuencias:

1. El 0,324 de Spearman **subestima** la relación, porque solo mide la componente monótona.
2. Una regresión logística con `Age` cruda fallará sistemáticamente en los mayores de 60:
   ajustará una curva creciente y les asignará riesgo alto cuando han vuelto al nivel de un
   cliente de 40. Este es el argumento —basado en evidencia, no en estética— para `age_group`.
3. Anticipa que los modelos de árbol superen a los lineales, porque particionan el espacio y
   capturan el pico sin necesidad de ayuda.

### La línea base de negocio

| Tramo | % de clientes | % del abandono total | Lift |
|---|---|---|---|
| 51-60 | 8,0% | 22,0% | **2,75×** |
| 41-50 | 23,2% | 38,7% | 1,67× |
| **41-60** | **31,2%** | **60,7%** | **1,95×** |

Contactando al 31% de los clientes —los de 41 a 60 años— se alcanza al **61% de todos los que van
a abandonar**, sin modelo alguno.

Esto redefine la vara de medir. El `DummyClassifier` demuestra que el accuracy no sirve, pero es
un rival de paja. La competencia real del modelo es esta regla de edad, porque es lo que el banco
puede hacer gratis y en una tarde. Se incorpora como **línea base de negocio** en el notebook 04.

### Decisiones tomadas

| # | Decisión | Fundamento |
|---|---|---|
| 1 | Conservar las 322 filas | No son imposibles; borrar el 3,2% apoyándose en un supuesto falso sería peor que el problema |
| 2 | No crear feature de incoherencia | Confusor demostrado: 7,0% vs 7,5% dentro del mismo tramo de edad |
| 3 | Eliminar `tenure_age_ratio` | El concepto que pretende medir no existe con una correlación de −0,01 |
| 4 | Añadir línea base de negocio | La regla de edad, con lift 1,95×, es el rival real del modelo |
| 5 | `age_group` justificada por evidencia | Los modelos lineales no pueden representar la U invertida |

**Hipótesis de negocio** — asociación, no causalidad: el pico en 51-60 corresponde a la antesala
de la jubilación (consolidación de productos, traspaso de planes de pensiones, comparación de
entidades ante una indemnización). A partir de los 60 predomina la inercia.

In [0]:
# Perfil de calidad "antes" — la rúbrica pide comparar antes/después
profile_before = pd.DataFrame({
    "dtype":     raw.dtypes.astype(str),
    "n_null":    raw.isna().sum(),
    "n_unique":  raw.nunique(),
    "ejemplo":   raw.iloc[0],
})
profile_before

,dtype,n_null,n_unique,ejemplo
RowNumber,int64,0,10000,1
CustomerId,int64,0,10000,15634602
Surname,object,0,2932,Hargrave
CreditScore,int64,0,460,619
Geography,object,0,3,France
Gender,object,0,2,Female
Age,int64,0,70,42
Tenure,int64,0,11,2
Balance,float64,0,6382,0.0
NumOfProducts,int64,0,4,1


## A3 · Contrato de datos: mapeo de nombres

El CSV trae PascalCase; las tablas usan snake_case (decisión 01). El mapeo es explícito y
vive en el código — el CSV **no se toca**, que es lo que el enunciado prohíbe.

Solo cambian los **nombres**. Ningún valor se altera en esta fase.

In [0]:
COLUMN_MAP = {
    "RowNumber":       "row_number",
    "CustomerId":      "customer_id",
    "Surname":         "surname",
    "CreditScore":     "credit_score",
    "Geography":       "geography",
    "Gender":          "gender",
    "Age":             "age",
    "Tenure":          "tenure",
    "Balance":         "balance",
    "NumOfProducts":   "num_of_products",
    "HasCrCard":       "has_cr_card",
    "IsActiveMember":  "is_active_member",
    "EstimatedSalary": "estimated_salary",
    "Exited":          "exited",
}

missing = set(COLUMN_MAP) - set(raw.columns)
extra   = set(raw.columns) - set(COLUMN_MAP)
assert not missing, f"columnas esperadas y ausentes: {missing}"
assert not extra,   f"columnas inesperadas en el origen: {extra}"

staged = raw.rename(columns=COLUMN_MAP)
staged["_source_file"] = TRACE["source_file"]
staged["_source_url"]  = TRACE["source_url"]

print("contrato verificado — 14 columnas mapeadas")
staged.head(3)

contrato verificado — 14 columnas mapeadas


,row_number,customer_id,surname,credit_score,geography,gender,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited,_source_file,_source_url
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,churn.csv,https://www.kaggle.com/datasets/mathchi/churn-...
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,churn.csv,https://www.kaggle.com/datasets/mathchi/churn-...
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,churn.csv,https://www.kaggle.com/datasets/mathchi/churn-...


## A4 · Carga a `bank_churn.customers_raw`

`TRUNCATE` antes de insertar hace la celda **idempotente**: reejecutarla no duplica filas.
Sin eso, dos ejecuciones dejarían 20.000 registros y ningún error visible.

Se usa `COPY ... FROM STDIN` en vez de `INSERT` fila a fila: una sola operación en lugar de
diez mil viajes de ida y vuelta.

In [0]:
COLS = list(COLUMN_MAP.values()) + ["_source_file", "_source_url"]

buf = io.StringIO()
staged[COLS].to_csv(buf, index=False, header=False)
buf.seek(0)

with connect() as conn:
    cur = conn.cursor()
    cur.execute(f"TRUNCATE TABLE {SCHEMA}.customers_raw;")
    cur.execute(
        f'COPY {SCHEMA}.customers_raw ({", ".join(COLS)}) FROM STDIN WITH (FORMAT CSV)',
        stream=buf,
    )
    conn.commit()

print("carga completada")

carga completada


In [0]:
# Evidencia exigida por la rúbrica: SELECT COUNT(*) contra la BD externa
print(query(f"SELECT COUNT(*) AS filas_cargadas FROM {SCHEMA}.customers_raw").to_string(index=False))
print()
print(query(f"""
    SELECT geography, COUNT(*) AS clientes,
           ROUND(AVG(exited)::numeric * 100, 2) AS tasa_abandono_pct
    FROM   {SCHEMA}.customers_raw
    GROUP  BY geography
    ORDER  BY clientes DESC
""").to_string(index=False))

 filas_cargadas
          10000

geography  clientes tasa_abandono_pct
   France      5014             16.15
  Germany      2509             32.44
    Spain      2477             16.67


La consulta de verificación deja ver ya el segundo patrón de negocio: **Alemania abandona al
32,4%, el doble que Francia (16,2%) y España (16,7%)**, y con la mitad de clientes que Francia.

Se explora a fondo en el notebook 03. Conviene anotar desde ahora que `Geography` es una de las
dos variables que el enunciado exige analizar **desde la perspectiva de sesgo**, así que este
hallazgo deberá revisarse también en clave de equidad —¿el modelo acierta igual de bien en los
tres países?— y no solo como oportunidad comercial.

## A5 · Tablas modeladas

**Pausa aquí.** Con los rangos reales de V4 a la vista, revisa las cotas de los `CHECK` en
`sql/02_modeled.sql`, ajústalas si hace falta, y ejecútalo en el **SQL Editor** de Lakebase.

El script SQL se mantiene como única fuente de verdad del esquema — es entregable
obligatorio del enunciado y no conviene tener una segunda copia dentro del notebook que
pueda desincronizarse.

Cuando esté ejecutado, la celda siguiente puebla `customers` desde `customers_raw`.

In [0]:
# Verificar si la tabla customers existe
table_exists = query(f"""
    SELECT EXISTS (
        SELECT 1 FROM information_schema.tables 
        WHERE table_schema = '{SCHEMA}' AND table_name = 'customers'
    ) AS exists
""").iloc[0, 0]

if not table_exists:
    print("ERROR: La tabla 'bank_churn.customers' no existe.")
    print("")
    print("ACCIÓN REQUERIDA:")
    print("  1. Abre el archivo 'sql/02_modeled.sql'")
    print("  2. Revisa las cotas de los CHECK constraints con los rangos de V4")
    print("  3. Ejecuta el script completo en el SQL Editor de Lakebase")
    print("  4. Vuelve a ejecutar esta celda")
    print("")
    print("Tablas disponibles en el schema:")
    print(query(f"""
        SELECT table_name FROM information_schema.tables 
        WHERE table_schema = '{SCHEMA}' ORDER BY table_name
    """).to_string(index=False))
else:
    # customers: registro operacional. No es la capa analítica (eso es Silver);
    # es el destino de la clave foranea de customer_predictions y la frontera de privacidad
    # donde surname deja de propagarse.
    POPULATE = f"""
    INSERT INTO {SCHEMA}.customers (
        customer_id, geography_id, gender, age, tenure, credit_score,
        balance, estimated_salary, num_of_products,
        has_cr_card, is_active_member, exited
    )
    SELECT r.customer_id,
           g.geography_id,
           r.gender,
           r.age,
           r.tenure,
           r.credit_score,
           r.balance,
           r.estimated_salary,
           r.num_of_products,
           r.has_cr_card      = 1,
           r.is_active_member = 1,
           r.exited           = 1
    FROM   {SCHEMA}.customers_raw r
    JOIN   {SCHEMA}.geographies  g ON g.country_name = r.geography
    ON CONFLICT (customer_id) DO NOTHING;
    """

    with connect() as conn:
        cur = conn.cursor()
        cur.execute(POPULATE)
        conn.commit()

    print(query(f"SELECT COUNT(*) AS clientes FROM {SCHEMA}.customers").to_string(index=False))

    # Prueba de la frontera de privacidad: surname no debe existir como columna
    cols = query("""
        SELECT column_name
        FROM   information_schema.columns
        WHERE  table_schema = %s AND table_name = 'customers'
        ORDER  BY ordinal_position
    """, (SCHEMA,))["column_name"].tolist()

    print("\ncolumnas de customers:", cols)
    assert "surname" not in cols, "surname se propago — revisar el INSERT"
    print("surname ausente — privacidad impuesta por el esquema, no por disciplina")

❌ ERROR: La tabla 'bank_churn.customers' no existe.

ACCIÓN REQUERIDA:
  1. Abre el archivo 'sql/02_modeled.sql'
  2. Revisa las cotas de los CHECK constraints con los rangos de V4
  3. Ejecuta el script completo en el SQL Editor de Lakebase
  4. Vuelve a ejecutar esta celda

Tablas disponibles en el schema:
   table_name
customers_raw
  geographies


---

# Parte B · Bronze

*Aquí empieza la arquitectura medallion.*

Bronze es la copia inmutable de la extracción JDBC. **No se transforma nada**: ni tipos, ni
booleanos, ni columnas derivadas. Todo eso ocurre en Silver.

El valor de Bronze es la trazabilidad — si Silver o Gold se corrompen, se reconstruyen desde
aquí sin volver a tocar el sistema operacional.

In [0]:
# Catalogo y esquema de destino en Unity Catalog
print(spark.sql("SHOW CATALOGS").toPandas().to_string(index=False))

    catalog
       itse
    samples
supermarket
     system
  workspace


In [0]:
CATALOG       = "bank_churn"   # creado en setup_entorno
BRONZE_SCHEMA = "bronze"
BRONZE_TABLE  = f"{CATALOG}.{BRONZE_SCHEMA}.bank_customers_raw"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
print("esquema listo:", f"{CATALOG}.{BRONZE_SCHEMA}")

esquema listo: workspace.bronze


In [0]:
token    = w.postgres.generate_database_credential(ENDPOINT).token
jdbc_url = f"jdbc:postgresql://{PGHOST}:{PGPORT}/{PGDATABASE}?sslmode=require"

bronze_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("driver", "org.postgresql.Driver")
    .option("dbtable", f"{SCHEMA}.customers_raw")
    .option("user", PGUSER)
    .option("password", token)
    .load()
)

print("esquema leído por JDBC:")
bronze_df.printSchema()
print("filas:", bronze_df.count())

esquema leido por JDBC:
root
 |-- row_number: integer (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- surname: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- geography: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- balance: decimal(14,2) (nullable = true)
 |-- num_of_products: integer (nullable = true)
 |-- has_cr_card: short (nullable = true)
 |-- is_active_member: short (nullable = true)
 |-- estimated_salary: decimal(14,2) (nullable = true)
 |-- exited: short (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_url: string (nullable = true)

filas: 10000


In [0]:
from pyspark.sql import functions as F

# Marca de extracción: distinta de _ingested_at, que registra cuando entro a Postgres
(
    bronze_df
    .withColumn("_extracted_at", F.current_timestamp())
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

spark.sql(f"""
    COMMENT ON TABLE {BRONZE_TABLE} IS
    'Bronze. Copia inmutable de la extracción JDBC de bank_churn.customers_raw (Lakebase).
     Sin transformaciones. Fuente de reconstrucción de Silver y Gold.'
""")

print("Bronze escrita:", BRONZE_TABLE)

Bronze escrita: workspace.bronze.bank_customers_raw


In [0]:
# Verificación de integridad: Bronze debe coincidir exactamente con el origen
n_pg     = int(query(f"SELECT COUNT(*) AS n FROM {SCHEMA}.customers_raw").iloc[0, 0])
n_bronze = spark.table(BRONZE_TABLE).count()

print(f"Postgres : {n_pg}")
print(f"Bronze   : {n_bronze}")
print("coinciden:", n_pg == n_bronze)

display(spark.table(BRONZE_TABLE).limit(5))

Postgres : 10000
Bronze   : 10000
coinciden: True


row_number,customer_id,surname,credit_score,geography,gender,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited,_ingested_at,_source_file,_source_url,_extracted_at
1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,2026-08-01T23:19:09.557Z,churn.csv,https://www.kaggle.com/datasets/mathchi/churn-for-bank-customers,2026-08-01T23:24:30.628Z
2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,2026-08-01T23:19:09.557Z,churn.csv,https://www.kaggle.com/datasets/mathchi/churn-for-bank-customers,2026-08-01T23:24:30.628Z
3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,2026-08-01T23:19:09.557Z,churn.csv,https://www.kaggle.com/datasets/mathchi/churn-for-bank-customers,2026-08-01T23:24:30.628Z
4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,2026-08-01T23:19:09.557Z,churn.csv,https://www.kaggle.com/datasets/mathchi/churn-for-bank-customers,2026-08-01T23:24:30.628Z
5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,2026-08-01T23:19:09.557Z,churn.csv,https://www.kaggle.com/datasets/mathchi/churn-for-bank-customers,2026-08-01T23:24:30.628Z


## Resultado

Cadena probada de extremo a extremo:

```
Kaggle ──seeding──▶ bank_churn.customers_raw ──JDBC──▶ bronze.bank_customers_raw
          (una vez)         (Lakebase)                      (Delta, 10.000 filas)
```

**Pendiente:** ejecutar `sql/02_modeled.sql` en el SQL Editor de Lakebase y volver a correr la
celda A5 para poblar `customers`. Las cotas de los `CHECK` se contrastaron contra V4 y **no
requieren ajuste**: todos los rangos observados caben dentro de las reglas de negocio definidas.

Nota deliberada: los `CHECK` no se ajustan a los rangos de la muestra. Un `credit_score BETWEEN
350 AND 850` rechazaría mañana a un cliente con 870 por un motivo que no existe. Una restricción
codifica una regla de negocio, no el recorrido de una muestra.

**Para el informe** — capturar de este notebook: el bloque de trazabilidad de A1 (con el hash
SHA-256), la tabla de verificaciones V1–V8, el diagnóstico A2.1, el `SELECT COUNT(*)` de A4 y el
`printSchema()` de Bronze.

**Siguiente:** `02_preparacion_datos` — auditoría de calidad antes/después, limpieza
determinística, features derivadas y tabla Silver. Primera decisión sobre la mesa: qué significa
`balance = 0`.